# Domain 3: Claude Code Configuration & Workflows
## TS 3.4: Determine When to Use Plan Mode vs Direct Execution
**Certification Focus**: CCAR-F Domain 3 (20% Exam Weight) | **Scenario Alignment**: Scenario 2 & 4

---

### 🎯 Key Architectural Principles
1. **Plan Mode Triggers**:
   - Multi-file architectural refactors (>3 files affected).
   - Tasks involving trade-offs between libraries, databases, or protocols.
   - Ambiguous or underspecified requirements.
   - High blast-radius operations where unintended edits would break dependent services.
2. **Direct Execution Triggers**:
   - Single-file isolated bug fixes.
   - Tasks accompanied by explicit stack traces with known root causes.
   - Simple refactors with pre-existing, comprehensive test coverage.
3. **Plan Mode Execution Contract**: In Plan Mode, Claude uses **read-only tools** (`Read`, `Grep`, `Glob`) to explore, formulates a numbered step-by-step plan, and asks the user for explicit approval before modifying code.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.4 Plan Mode vs Direct Execution.")

### ⚖️ Step 2: Mode Evaluation Engine & Interview Prompts
We implement the decision logic evaluating whether a task warrants Plan Mode or Direct Execution.

In [ ]:
# Step 2: Mode Evaluation Logic
@dataclass
class WorkflowStrategyDecision:
    mode: str  # 'plan_mode' | 'direct_execution'
    rationale: str
    recommended_tools: List[str]


def evaluate_execution_mode(
    file_count_affected: int,
    has_architectural_tradeoffs: bool,
    has_known_stack_trace: bool,
    is_single_function_fix: bool,
) -> WorkflowStrategyDecision:
    """Determines whether to trigger Plan Mode vs Direct Execution."""
    if has_architectural_tradeoffs or file_count_affected > 3 or (not has_known_stack_trace and not is_single_function_fix):
        return WorkflowStrategyDecision(
            mode="plan_mode",
            rationale="Complex task involving multi-file impact or architectural trade-offs requires Plan Mode.",
            recommended_tools=["Read", "Grep", "Glob", "Explore"],
        )
    return WorkflowStrategyDecision(
        mode="direct_execution",
        rationale="Isolated single-file fix with clear stack trace is suitable for Direct Execution.",
        recommended_tools=["Read", "Edit", "Bash"],
    )

def build_interview_pattern_prompt(task_description: str) -> str:
    """Generates prompt instructing Claude to interview user before planning."""
    return (
        f"You are in Plan Mode for task: '{task_description}'. "
        "Do NOT write code or modify files yet. "
        "First, interview me by asking 2-3 clarifying questions with concrete trade-off options."
    )

print(" Mode evaluation engine initialized.")

### 🧪 Step 3: Interactive Simulation
1. Complex refactor -> Plan Mode.
2. Isolated bugfix -> Direct Execution.

In [ ]:
# Step 3: Simulation
# Scenario 1: Multi-file caching refactor
dec_plan = evaluate_execution_mode(
    file_count_affected=8,
    has_architectural_tradeoffs=True,
    has_known_stack_trace=False,
    is_single_function_fix=False,
)
print("Scenario 1 Mode:", dec_plan.mode)
print("Scenario 1 Rationale:", dec_plan.rationale)
print("Allowed Tools in Plan Mode:", dec_plan.recommended_tools)

# Scenario 2: Isolated ZeroDivisionError bugfix
dec_direct = evaluate_execution_mode(
    file_count_affected=1,
    has_architectural_tradeoffs=False,
    has_known_stack_trace=True,
    is_single_function_fix=True,
)
print("\nScenario 2 Mode:", dec_direct.mode)
print("Scenario 2 Rationale:", dec_direct.rationale)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Multi-file refactor strictly routes to Plan Mode with read-only tools.
2. Single-file fix routes to Direct Execution.
3. Interview prompt instructs Claude to interview the user.

In [ ]:
# Step 4: Verification Checks
assert dec_plan.mode == "plan_mode"
assert "Explore" in dec_plan.recommended_tools or "Read" in dec_plan.recommended_tools
assert "Edit" not in dec_plan.recommended_tools, "Plan mode must use read-only exploration!"

assert dec_direct.mode == "direct_execution"
assert "Edit" in dec_direct.recommended_tools

interview_prompt = build_interview_pattern_prompt("Cache layer")
assert "interview me" in interview_prompt

print(" All verification assertions passed! Mode evaluation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use **Plan Mode** for tasks touching >3 files, architectural redesigns, or ambiguous requirements.
- **Rule 2**: In Plan Mode, use **read-only tools** (`Read`, `Grep`, `Glob`) to explore; do not edit files without plan approval.
- **Rule 3**: Use **Direct Execution** only for single-file, low blast-radius bug fixes with clear stack traces.